In [1]:
import pandas as pd
import geopandas as gpd
import requests as re
from io import BytesIO
from os import path, makedirs
from unidecode import unidecode

# Produção de habitação de interesse social

Vários indicadores do GT Urbanismo são relacionados à produção e entrega de unidades de habitação de interesse social. O formulário 15 é relacionado ao orçamento utilizado na produção de unidades de habitação de interesse social. O primeiro indicador físico está no formulário 18 e diz respeito à meta de provimento de moradias do PdM 2021-2024. O formulário 20 é relacionado a uma iniciativa específica do PdM, mas os dados de execução estão disponíveis apenas no relatório da Função Habitação. Por último, os formulários 19 e 21 são relacionado, mas não possuem meta no PdM, apenas nos ODS da Agenda 2030.

Posteriormente, um novo documento apresentou várias informações diferentes relacionadas ao orçamento da função habitação, onde está incluso o Programa 3002. Portanto, vamos manter todo o orçamento da função habitação e apresentar os diferentes recortes no Qlik Sense.

## ObservaSampa

Como vamos usar vários indicadores do ObservaSampa parece fazer sentido usar uma função genérica para download de indicadores do ObservaSampa.

In [2]:
indicadores_observa = [
    {
        "nome_completo": "11.01.02 Número de famílias beneficiadas por procedimentos de regularização fundiária em núcleos urbanos informais",
        "nome_curto": "indicador_110102",
        "id_observa": 283,
        "nome_coluna_valor": "qtd_familias"
    },

    {
        "nome_completo": "11.01.06 Número de famílias beneficiadas com obras de urbanização de assentamentos precários",
        "nome_curto": "indicador_110106",
        "id_observa": 402,
        "nome_coluna_valor": "qtd_familias"
    }, 

    {
        "nome_completo": "11.01.07 Número estimado de domicílios em favelas",
        "nome_curto": "indicador_110107",
        "id_observa": 288,
        "nome_coluna_valor": "qtd_domicilios"
    },

    {
        "nome_completo": "Número de termos de Permissão de Uso (TPU) emitidos em nome da mulher da familia",
        "nome_curto": "indicador_tpu",
        "id_observa": 662,
        "nome_coluna_valor": "qtd_termos"
    },
    {
        "nome_completo": "Produção de habitação de interesse social",
        "nome_curto": "indicador_producao",
        "id_observa": 285,
        "nome_coluna_valor": "qtd_unidades"
    },
    {
        "nome_completo": "01.05.02 Número de famílias em atendimento habitacional provisório (Auxílio Aluguel) por situação de risco e emergência",
        "nome_curto": "indicador_010502",
        "id_observa": 660,
        "nome_coluna_valor": "qtd_familias"
    },
    {
        "nome_completo": "11.01.08 Total de famílias beneficiárias de Auxílio Aluguel por ano",
        "nome_curto": "indicador_110108",
        "id_observa": 547,
        "nome_coluna_valor": "qtd_familias"
    },
    {
        "nome_completo": "Estimativa de domicílios em favela (%)",
        "nome_curto": "indicador_edf",
        "id_observa": 286,
        "nome_coluna_valor": "qtd_domicilios"
    }
]

In [3]:
len(indicadores_observa)

8

In [4]:
def extrair_observa(id_observa:int) -> pd.DataFrame:
    url_observa = f'https://observasampa.prefeitura.sp.gov.br/arquivo.php?cd_indicador={id_observa}'
    df_observa = pd.read_csv(url_observa, sep=';', decimal=',', encoding='utf8')
    return df_observa

In [5]:

def padronizar_subprefeitura(regiao: str) -> str:
    if pd.isna(regiao): return 'Sem informação'
    return unidecode(regiao).strip().lower()

In [6]:
def filtrar_nivel_regional(df: pd.DataFrame, indicador_regionalizado: bool) -> pd.DataFrame:
    if indicador_regionalizado:
        df = df[df['nivel_regional'] == 'Subprefeitura']
        df = df.rename(columns={'região': 'subprefeitura'})
    else:
        df = df[df['nivel_regional'] == 'Município']
        df = df.drop(columns=['região'])
    df = df.drop(columns=['nivel_regional'])
    return df

In [7]:
def despivotar_anos(df: pd.DataFrame,
                nome_coluna_valor:str,
                anos: list[str]=['2022', '2023', '2024']) -> pd.DataFrame:
    
    # Detectar colunas ID dinamicamente
    id_cols = [col for col in ['subprefeitura', 'indicador'] if col in df.columns]
    
    df = df.melt(
        id_vars=id_cols,
        value_vars=[col for col in df.columns if col in anos],
        var_name='ano',
        value_name=nome_coluna_valor
    )
    return df

In [8]:
def pipeline_completa(indicador: str, anos: list[str]=['2022', '2023', '2024']) -> pd.DataFrame:
    indicador_info = next((item for item in indicadores_observa if item['nome_curto'] == indicador), None)
    
    df_bom = extrair_observa(indicador_info['id_observa'])

    if df_bom['nivel_regional'].str.lower().str.contains('subprefeitura').any():
        df_bom = filtrar_nivel_regional(df_bom, indicador_regionalizado=True)
    else:
        df_bom = filtrar_nivel_regional(df_bom, indicador_regionalizado=False)

    df_bom = despivotar_anos(df_bom,
                                nome_coluna_valor=indicador_info['nome_coluna_valor'],
                                anos=anos)

    return df_bom

### 11.01.02 Número de famílias beneficiadas por procedimentos de regularização fundiária em núcleos urbanos informais


In [9]:

df_110102 = pipeline_completa('indicador_110102', anos=['2024', '2025'])
df_110102

,subprefeitura,indicador,ano,qtd_familias
0,Vila Mariana,11.01.02 Número de famílias beneficiadas por p...,2024,93.0
1,Penha,11.01.02 Número de famílias beneficiadas por p...,2024,1842.0
2,Perus,11.01.02 Número de famílias beneficiadas por p...,2024,4042.0
3,Sapopemba,11.01.02 Número de famílias beneficiadas por p...,2024,6233.0
4,Vila Prudente,11.01.02 Número de famílias beneficiadas por p...,2024,49.0
...,...,...,...,...
59,Vila Maria-Vila Guilherme,11.01.02 Número de famílias beneficiadas por p...,2025,4.0
60,Lapa,11.01.02 Número de famílias beneficiadas por p...,2025,96.0
61,Jaçanã-Tremembé,11.01.02 Número de famílias beneficiadas por p...,2025,1061.0
62,Santana-Tucuruvi,11.01.02 Número de famílias beneficiadas por p...,2025,0.0


### 01.05.02 Número de famílias em atendimento habitacional provisório (Auxílio Aluguel) por situação de risco e emergência

In [10]:
df_010502 = pipeline_completa('indicador_010502')
df_010502

,subprefeitura,indicador,ano,qtd_familias
0,Vila Maria-Vila Guilherme,01.05.02 Número de famílias em atendimento hab...,2022,38.0
1,Santana-Tucuruvi,01.05.02 Número de famílias em atendimento hab...,2022,0.0
2,Pirituba-Jaraguá,01.05.02 Número de famílias em atendimento hab...,2022,167.0
3,Santo Amaro,01.05.02 Número de famílias em atendimento hab...,2022,504.0
4,Itaim Paulista,01.05.02 Número de famílias em atendimento hab...,2022,20.0
...,...,...,...,...
91,M'Boi Mirim,01.05.02 Número de famílias em atendimento hab...,2024,1312.0
92,São Mateus,01.05.02 Número de famílias em atendimento hab...,2024,416.0
93,São Miguel,01.05.02 Número de famílias em atendimento hab...,2024,122.0
94,Sapopemba,01.05.02 Número de famílias em atendimento hab...,2024,230.0


### 11.01.06 Número de famílias beneficiadas com obras de urbanização de assentamentos precários

In [11]:
df_110106 = pipeline_completa('indicador_110106')
df_110106

,subprefeitura,indicador,ano,qtd_familias
0,Pirituba-Jaraguá,11.01.06 Número de famílias beneficiadas com o...,2022,0.0
1,Lapa,11.01.06 Número de famílias beneficiadas com o...,2022,0.0
2,M'Boi Mirim,11.01.06 Número de famílias beneficiadas com o...,2022,3006.0
3,Mooca,11.01.06 Número de famílias beneficiadas com o...,2022,0.0
4,Parelheiros,11.01.06 Número de famílias beneficiadas com o...,2022,33.0
...,...,...,...,...
91,Santana-Tucuruvi,11.01.06 Número de famílias beneficiadas com o...,2024,0.0
92,Santo Amaro,11.01.06 Número de famílias beneficiadas com o...,2024,0.0
93,São Mateus,11.01.06 Número de famílias beneficiadas com o...,2024,2776.0
94,São Miguel,11.01.06 Número de famílias beneficiadas com o...,2024,0.0


### 11.01.07 Número estimado de domicílios em favelas

In [12]:
df_110107 = pipeline_completa('indicador_110107')
df_110107

,subprefeitura,indicador,ano,qtd_domicilios
0,Lapa,11.01.07 Número estimado de domicílios em favelas,2022,2825.0
1,Vila Prudente,11.01.07 Número estimado de domicílios em favelas,2022,4155.0
2,Aricanduva-Formosa-Carrão,11.01.07 Número estimado de domicílios em favelas,2022,1338.0
3,Sé,11.01.07 Número estimado de domicílios em favelas,2022,463.0
4,M'Boi Mirim,11.01.07 Número estimado de domicílios em favelas,2022,43566.0
...,...,...,...,...
91,Casa Verde-Cachoeirinha,11.01.07 Número estimado de domicílios em favelas,2024,11549.0
92,Cidade Ademar,11.01.07 Número estimado de domicílios em favelas,2024,25539.0
93,Guaianases,11.01.07 Número estimado de domicílios em favelas,2024,5054.0
94,Jabaquara,11.01.07 Número estimado de domicílios em favelas,2024,12983.0


### 11.01.08 Total de famílias beneficiárias de Auxílio Aluguel por ano

In [13]:
df_110108 = pipeline_completa('indicador_110108')
df_110108

,subprefeitura,indicador,ano,qtd_familias
0,Itaim Paulista,11.01.08 Total de famílias beneficiárias de Au...,2022,20.0
1,Parelheiros,11.01.08 Total de famílias beneficiárias de Au...,2022,50.0
2,São Mateus,11.01.08 Total de famílias beneficiárias de Au...,2022,909.0
3,São Miguel,11.01.08 Total de famílias beneficiárias de Au...,2022,180.0
4,Ermelino Matarazzo,11.01.08 Total de famílias beneficiárias de Au...,2022,8.0
...,...,...,...,...
91,Cidade Tiradentes,11.01.08 Total de famílias beneficiárias de Au...,2024,90.0
92,Guaianases,11.01.08 Total de famílias beneficiárias de Au...,2024,20.0
93,Pirituba-Jaraguá,11.01.08 Total de famílias beneficiárias de Au...,2024,173.0
94,Sapopemba,11.01.08 Total de famílias beneficiárias de Au...,2024,489.0


### 11.01.03 Produção de habitação de interesse social

In [14]:
df_his = pipeline_completa('indicador_producao')
df_his

,subprefeitura,indicador,ano,qtd_unidades
0,Butantã,11.01.03 Número de Unidades Habitacionais entr...,2022,0.0
1,Jabaquara,11.01.03 Número de Unidades Habitacionais entr...,2022,0.0
2,São Miguel,11.01.03 Número de Unidades Habitacionais entr...,2022,0.0
3,Jaçanã-Tremembé,11.01.03 Número de Unidades Habitacionais entr...,2022,0.0
4,Lapa,11.01.03 Número de Unidades Habitacionais entr...,2022,181.0
...,...,...,...,...
91,Aricanduva-Formosa-Carrão,11.01.03 Número de Unidades Habitacionais entr...,2024,0.0
92,Freguesia-Brasilândia,11.01.03 Número de Unidades Habitacionais entr...,2024,0.0
93,Pirituba-Jaraguá,11.01.03 Número de Unidades Habitacionais entr...,2024,5.0
94,Cidade Ademar,11.01.03 Número de Unidades Habitacionais entr...,2024,1.0


O indicador do formulário 19 não está ligado diretamente a nenhuma meta específica do PdM, mas está em consonância com o indicador 11.01.03 dos ODS da Agenda 2030. Os dados desse indicador estão disponíveis no [Observasampa](https://observasampa.prefeitura.sp.gov.br/).

### 05.0a.04 Número de termos de Permissão de Uso (TPU) emitidos em nome da mulher da familia

Esse indicador possui é um exemplo da transversalidade de gênero, que integra a perspectiva de gênero na construção da politica pública voltada à provisão de habitação de interesse social. Os dados desse indicador estão disponíveis, de maneira não regionalizada, no [Observasampa](https://observasampa.prefeitura.sp.gov.br/), no indicador 05.0a.04. Posteriormente, os dados regionalizados estarão presentes no relatório da Função Habitação.

In [15]:
df_tpu = pipeline_completa('indicador_tpu')
df_tpu

,indicador,ano,qtd_termos
0,05.0a.04 Número de termos de Permissão de Uso ...,2022,293.0
1,05.0a.04 Número de termos de Permissão de Uso ...,2023,384.0
2,05.0a.04 Número de termos de Permissão de Uso ...,2024,458.0


### Estimativa de domicílios em favela (%)

In [16]:
df_edf = pipeline_completa('indicador_edf')
df_edf

,subprefeitura,indicador,ano,qtd_domicilios
0,Santo Amaro,Estimativa de domicílios em favela (%),2022,4.2877
1,M'Boi Mirim,Estimativa de domicílios em favela (%),2022,19.5985
2,Mooca,Estimativa de domicílios em favela (%),2022,1.0496
3,Vila Prudente,Estimativa de domicílios em favela (%),2022,4.7332
4,Guaianases,Estimativa de domicílios em favela (%),2022,5.3583
5,Ipiranga,Estimativa de domicílios em favela (%),2022,14.8267
6,Penha,Estimativa de domicílios em favela (%),2022,6.4167
7,Pinheiros,Estimativa de domicílios em favela (%),2022,0.2640
8,Pirituba-Jaraguá,Estimativa de domicílios em favela (%),2022,10.6484
9,Freguesia-Brasilândia,Estimativa de domicílios em favela (%),2022,17.6342


# Exportando os arquivos

Neste notebook, vamos apenas salvar os arquivos extraídos na pasta de entrada de dados.

In [17]:
output_dir = path.join('data', 'cache', 'urbanismo')
if not path.exists(output_dir):
    makedirs(output_dir)

for nome, df in [
    ('his_entregue_original', df_his),
    ('tpu_emitido_original', df_tpu),
    ('indicador_110107', df_110107),
    ('indicador_110102', df_110102),
    ('indicador_110106', df_110106),
    ('indicador_010502', df_010502),
    ('indicador_110108', df_110108),
    ('indicador_estimativa_de_domicilios_em_favela', df_edf),
]:
    filename = path.join(output_dir, nome)
    df.to_csv(f'{filename}.csv', sep=';', decimal=',', encoding='utf8', index=False)
